# Market (S&P 500 daily closes): the edge that was too easy

*Notebook journey for §3 of the dissertation. Every number the chapter uses is produced here. This is where one feature row is built from the raw prices one transform at a time, printed line by line, so we can see exactly what the model is fed.*

## a. Reading the problem

In [1]:
import numpy as np, os

# the market data (S&P 500 daily closes): ONE column of prices, in date order
DATA = "data/gspc_2026-07-03.csv" if os.path.exists("data/gspc_2026-07-03.csv") else "../data/gspc_2026-07-03.csv"
close = np.loadtxt(DATA, skiprows=1)
print("prices:", len(close), "(a single column, in date order)")
print("first six closes:", [round(float(v), 2) for v in close[:6]])
print("the index ranges from", round(float(close.min())), "to", round(float(close.max())), "over the years")

# build the task one transform at a time, and TRACE the first few days
r = np.diff(close) / close[:-1]        # daily return
s = np.abs(r)                          # size of the move (a volatility proxy)
print("\ntrace, day by day:")
for t in range(1, 6):
    print(f"  day {t}: close {close[t]:.2f}  ->  return {r[t-1]:+.4f}  ->  size |r| {s[t-1]:.4f}")

# a feature row = the last five sizes; its label = does the NEXT size beat the median?
L = 5
X = np.column_stack([s[i:len(s) - L + i] for i in range(L)])
fut = s[L:]; X = X[:len(fut)]
med = float(np.median(s))
y = (fut > med).astype(int)
print(f"\nmedian move size: {med:.4f}   (the label threshold)")
print("one feature row X[0] = the five sizes above:", [round(float(v), 4) for v in X[0]])
print("          its label = next day's size beats the median?", int(y[0]))
print(f"\nrows: {len(y)} | classes balanced at {y.mean():.3f} | lag-1 autocorr of size: {np.corrcoef(s[:-1], s[1:])[0, 1]:.3f}")

prices: 6664 (a single column, in date order)
first six closes: [1455.22, 1399.42, 1402.11, 1403.45, 1441.47, 1457.6]
the index ranges from 677 to 7610 over the years

trace, day by day:
  day 1: close 1399.42  ->  return -0.0383  ->  size |r| 0.0383
  day 2: close 1402.11  ->  return +0.0019  ->  size |r| 0.0019
  day 3: close 1403.45  ->  return +0.0010  ->  size |r| 0.0010
  day 4: close 1441.47  ->  return +0.0271  ->  size |r| 0.0271
  day 5: close 1457.60  ->  return +0.0112  ->  size |r| 0.0112

median move size: 0.0054   (the label threshold)
one feature row X[0] = the five sizes above: [0.0383, 0.0019, 0.001, 0.0271, 0.0112]
          its label = next day's size beats the median? 1

rows: 6658 | classes balanced at 0.500 | lag-1 autocorr of size: 0.287


## b. The edge that was too easy

First the obvious try, direction: up or down tomorrow? Then a better question: the size of tomorrow's move. We split each one two ways, the book's shuffle and the honest past-to-future cut, and watch whether they agree. Direction also serves as a control at the end.

In [2]:
# the model (same small network from Section 1) + helpers
def init(d, width, K, rng):
    return [rng.standard_normal((d, width)) * 0.1, np.zeros(width),
            rng.standard_normal((width, K)) * 0.1, np.zeros(K)]
def forward(Xb, p):
    W1, b1, W2, b2 = p
    a = Xb @ W1 + b1; h = np.maximum(a, 0.0); z = h @ W2 + b2
    return a, h, z
def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def accuracy(p, Xb, yb):
    return float((forward(Xb, p)[2].argmax(1) == yb).mean())
def train(Xtr, ytr, width, lr, epochs, K, rng):
    p = init(Xtr.shape[1], width, K, rng); n = len(ytr)
    Y = np.zeros((n, K)); Y[np.arange(n), ytr] = 1.0
    for _ in range(epochs):
        a, h, z = forward(Xtr, p); dz = (softmax(z) - Y) / n
        W1, b1, W2, b2 = p
        dW2 = h.T @ dz; db2 = dz.sum(0); da = (dz @ W2.T) * (a > 0)
        dW1 = Xtr.T @ da; db1 = da.sum(0)
        p = [W1-lr*dW1, b1-lr*db1, W2-lr*dW2, b2-lr*db2]
    return p
def standardize(Xtr, Xte):
    m, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    return (Xtr - m) / sd, (Xte - m) / sd

def load(target="volatility", L=5):
    ret = np.diff(close) / close[:-1]; sz = np.abs(ret)
    feats = sz if target == "volatility" else ret
    Xf = np.column_stack([feats[i:len(feats) - L + i] for i in range(L)])
    if target == "volatility":
        fut = sz[L:]; Xf = Xf[:len(fut)]; yf = (fut > np.median(sz)).astype(int)
    else:
        fut = ret[L:]; Xf = Xf[:len(fut)]; yf = (fut > 0).astype(int)
    return Xf, yf

def score(Xf, yf, how, seed, lr=0.5):
    n = len(yf); ntest = n // 5
    if how == "chronological":                 # RIGHT: past -> train, future -> test
        tr, te = np.arange(n - ntest), np.arange(n - ntest, n)
    else:                                      # WRONG: shuffle, the future leaks into train
        idx = np.random.default_rng(seed).permutation(n); te, tr = idx[:ntest], idx[ntest:]
    Xtr, Xte = standardize(Xf[tr], Xf[te])
    p = train(Xtr, yf[tr], 16, lr, 300, 2, np.random.default_rng(seed + 1))
    return accuracy(p, Xte, yf[te])

def walk_forward(Xf, yf, folds=5):
    n = len(yf); bs = n // (folds + 1); out = []
    for k in range(1, folds + 1):
        tr, te = np.arange(0, k * bs), np.arange(k * bs, (k + 1) * bs)
        Xtr, Xte = standardize(Xf[tr], Xf[te])
        p = train(Xtr, yf[tr], 16, 0.5, 300, 2, np.random.default_rng(k))
        out.append(accuracy(p, Xte, yf[te]))
    return float(np.mean(out))

Xv, yv = load("volatility")
Xd, yd = load("direction")

d_sh = np.mean([score(Xd, yd, "shuffle", s) for s in range(5)]); d_ch = score(Xd, yd, "chronological", 0)
v_sh = np.mean([score(Xv, yv, "shuffle", s) for s in range(10)]); v_ch = score(Xv, yv, "chronological", 0)
v_wf = walk_forward(Xv, yv)

print("DIRECTION (up or down tomorrow?):")
print("  shuffle", round(float(d_sh), 3), " chronological", round(d_ch, 3), "  -> a coin flip, dead end")
print("\nVOLATILITY (busy day tomorrow?):")
print("  shuffle (by the book) :", round(float(v_sh), 3))
print("  chronological (honest):", round(v_ch, 3), "  <- they disagree")
print("  walk-forward (rolling):", round(v_wf, 3))
print("\nwhy: consecutive feature rows overlap")
print("  X[0]:", [round(float(v), 4) for v in Xv[0]])
print("  X[1]:", [round(float(v), 4) for v in Xv[1]], " (same four middle numbers, shifted by one)")
print("\ncontrol: on DIRECTION, a known coin, shuffle and honest agree, so the")
print("shuffle only inflates a score where there is a real signal to steal.")

DIRECTION (up or down tomorrow?):
  shuffle 0.533  chronological 0.54   -> a coin flip, dead end

VOLATILITY (busy day tomorrow?):
  shuffle (by the book) : 0.618
  chronological (honest): 0.587   <- they disagree
  walk-forward (rolling): 0.603

why: consecutive feature rows overlap
  X[0]: [0.0383, 0.0019, 0.001, 0.0271, 0.0112]
  X[1]: [0.0019, 0.001, 0.0271, 0.0112, 0.0131]  (same four middle numbers, shifted by one)

control: on DIRECTION, a known coin, shuffle and honest agree, so the
shuffle only inflates a score where there is a real signal to steal.


## c. A working model, killed by a unit

We scaled by the book: freeze the training mean and spread, apply them to the test. Now change one innocent thing, the feature's unit, from percent to raw points, keep the honest chronological split, and watch what happens.

In [3]:
# Trail B: keep the honest chronological split, change only the feature's UNIT and the scaler.
# the label is always built from the percent size, so it stays stationary and comparable.
def load_scaled(kind="percent", L=5):
    ret = np.diff(close) / close[:-1]
    a = np.abs(ret)                                            # percent size: stationary
    src = a if kind == "percent" else np.abs(np.diff(close))    # points size: drifts with the level
    Xf = np.column_stack([src[i:len(src) - L + i] for i in range(L)])
    fut = a[L:]; Xf = Xf[:len(fut)]
    yf = (fut > np.median(a)).astype(int)                      # label ALWAYS from the percent size
    return Xf, yf

def frozen(Xtr, Xte):                                          # textbook: fit on train, freeze, apply
    m, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    return (Xtr - m) / sd, (Xte - m) / sd

def rolling_z(Xf, W=250):                                      # drift-aware: each row scaled by its own trailing window
    Z = np.zeros_like(Xf)
    for j in range(Xf.shape[1]):
        c = Xf[:, j]; cs = np.concatenate([[0.0], np.cumsum(c)]); cs2 = np.concatenate([[0.0], np.cumsum(c * c)])
        for t in range(len(c)):
            lo = max(0, t - W); m = t - lo
            if m < 30:
                continue
            mu = (cs[t] - cs[lo]) / m
            var = (cs2[t] - cs2[lo]) / m - mu * mu
            Z[t, j] = (c[t] - mu) / (np.sqrt(max(var, 1e-18)) + 1e-9)
    return Z

def chron(n):
    ntest = n // 5
    return np.arange(n - ntest), np.arange(n - ntest, n)

def score_scaling(kind, scaler):
    Xf, yf = load_scaled(kind); n = len(yf); tr, te = chron(n)
    Xtr, Xte = (frozen(Xf[tr], Xf[te]) if scaler == "frozen" else (rolling_z(Xf)[tr], rolling_z(Xf)[te]))
    accs = [accuracy(train(Xtr, yf[tr], 16, 0.5, 300, 2, np.random.default_rng(s)), Xte, yf[te]) for s in range(10)]
    return float(np.mean(accs))

print("under the frozen rule, how far outside the training range does the test land?")
for kind in ("percent", "dollars"):
    Xf, yf = load_scaled(kind); n = len(yf); tr, te = chron(n)
    _, Xte = frozen(Xf[tr], Xf[te])
    print(f"  {kind:8s}: test |z| mean {np.abs(Xte).mean():.2f}   max {np.abs(Xte).max():.2f}")

print("\nsame chronological split, only the feature unit and the scaling change:")
print("  percent (stationary), frozen :", round(score_scaling("percent", "frozen"), 3))
print("  points  (drifts),     frozen :", round(score_scaling("dollars", "frozen"), 3))
print("  points  (drifts),     rolling:", round(score_scaling("dollars", "rolling"), 3))

under the frozen rule, how far outside the training range does the test land?
  percent : test |z| mean 0.57   max 9.19
  dollars : test |z| mean 1.67   max 27.69

same chronological split, only the feature unit and the scaling change:


  percent (stationary), frozen : 0.586


  points  (drifts),     frozen : 0.51


  points  (drifts),     rolling: 0.555
